# Parallel scan (prefix sum) on the GPU: chapter 11

In session 09 you parallelized a prefix sum on the CPU with `std::async` and three phases (`09_task_parallelism/example_54.cpp`, `exercise_9_2.cpp`). This folder does the same thing on a GPU. There are thousands of threads instead of eight, so the question changes. It is no longer "how do I split the array into a few chunks" but "how do 256 threads that share a small scratchpad scan their piece in a few steps, and how do thousands of such pieces agree on their offsets".

This notebook goes **step by step**: 7 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/09_scan/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 7: `01_sequential_scan.cu`

The sequential inclusive and exclusive scan of 8 values, run by a single GPU thread: n − 1 additions and nothing in parallel yet.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_sequential_scan.cu
#include <cstdio>
#include <numeric>
#include <vector>

#include "cuda_compat.h"

__global__ void sequential_inclusive_scan(const int* X, int* Y, int n) {
  int run = 0;
  for (int i = 0; i < n; ++i) {
    run += X[i];
    Y[i] = run;
  }
}

__global__ void sequential_exclusive_scan(const int* X, int* Y, int n) {
  int run = 0;
  for (int i = 0; i < n; ++i) {
    Y[i] = run;
    run += X[i];
  }
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-10s", label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

int main() {
  const std::vector<int> x = {3, 1, 7, 0, 4, 1, 6, 3};
  const int n = static_cast<int>(x.size());

  int* d_x = deviceCopyOf(x);
  int* d_inclusive = deviceCopyOf(std::vector<int>(n));
  int* d_exclusive = deviceCopyOf(std::vector<int>(n));

  launch(sequential_inclusive_scan, dim3(1), dim3(1), d_x, d_inclusive, n);
  launch(sequential_exclusive_scan, dim3(1), dim3(1), d_x, d_exclusive, n);
  CHECK(cudaGetLastError());

  const std::vector<int> inclusive = hostCopyOf(d_inclusive, n);
  const std::vector<int> exclusive = hostCopyOf(d_exclusive, n);
  printRow("x", x);
  printRow("inclusive", inclusive);
  printRow("exclusive", exclusive);

  std::vector<int> expectedInclusive(n);
  std::vector<int> expectedExclusive(n);
  std::inclusive_scan(x.begin(), x.end(), expectedInclusive.begin());
  std::exclusive_scan(x.begin(), x.end(), expectedExclusive.begin(), 0);
  const bool pass = (inclusive == expectedInclusive) && (exclusive == expectedExclusive);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_x));
  CHECK(cudaFree(d_inclusive));
  CHECK(cudaFree(d_exclusive));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_sequential_scan.cu -o 01_sequential_scan && ./01_sequential_scan

**What to look at in the output:** The exclusive row is the inclusive row shifted one place to the right with 0 in front: inclusive[2] = 11 is 3+1+7, and exclusive[3] is the same 11 because element 3 itself is left out. The last inclusive value, 25, is the total of all 8 values, reached after 7 additions in a row.

## Step 2 of 7: `02_kogge_stone.cu`

Kogge-Stone in one block, one thread per element, two barriers per step (read, wait, write). The array is printed after each stride; the barrier before each print is only there for the printing.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_kogge_stone.cu
#include <cstdio>
#include <numeric>
#include <vector>

#include "cuda_compat.h"

constexpr int SECTION_SIZE = 8;

__device__ void print_after_stride(int stride, const int* XY) {
  __syncthreads();
  if (threadIdx.x == 0) {
    printf("  stride %-3d", stride);
    for (int j = 0; j < SECTION_SIZE; ++j) {
      printf("%4d", XY[j]);
    }
    printf("\n");
  }
}

__global__ void kogge_stone_scan(const int* X, int* Y, int n) {
  __shared__ int XY[SECTION_SIZE];
  const int t = threadIdx.x;
  const int i = blockIdx.x * SECTION_SIZE + t;
  XY[t] = (i < n) ? X[i] : 0;

  for (int stride = 1; stride < SECTION_SIZE; stride *= 2) {
    __syncthreads();
    int temp = XY[t];
    if (t >= stride) {
      temp += XY[t - stride];
    }
    __syncthreads();
    XY[t] = temp;
    print_after_stride(stride, XY);
  }

  if (i < n) {
    Y[i] = XY[t];
  }
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-10s", label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

int main() {
  const std::vector<int> x = {3, 1, 7, 0, 4, 1, 6, 3};
  const int n = static_cast<int>(x.size());

  int* d_x = deviceCopyOf(x);
  int* d_y = deviceCopyOf(std::vector<int>(n));

  printRow("x", x);
  launch(kogge_stone_scan, dim3(1), dim3(SECTION_SIZE), d_x, d_y, n);
  CHECK(cudaGetLastError());
  const std::vector<int> y = hostCopyOf(d_y, n);
  printRow("inclusive", y);

  std::vector<int> expected(n);
  std::inclusive_scan(x.begin(), x.end(), expected.begin());
  const bool pass = (y == expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_x));
  CHECK(cudaFree(d_y));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_kogge_stone.cu -o 02_kogge_stone && ./02_kogge_stone

**What to look at in the output:** After stride 1 every position holds the sum of up to 2 elements (position 2 = 1+7 = 8), after stride 2 up to 4 (position 7 = 4+1+6+3 = 14), and after stride 4 the whole prefix. Positions below the stride no longer change: 3 4 11 11 is already final after stride 2. Try: change the loop to `stride < 4`; the run stops after stride 2, positions 4 to 7 keep the partial sums 12 12 11 14 and the check fails.

## Step 3 of 7: `03_kogge_stone_double_buffer.cu`

Two shared buffers that swap roles, so one barrier per step is enough. Each stride prints the buffer it reads, which nobody writes in that step.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_kogge_stone_double_buffer.cu
#include <cstdio>
#include <numeric>
#include <vector>

#include "cuda_compat.h"

constexpr int SECTION_SIZE = 8;

__device__ void print_buffer_read_by(int stride, int src, const int* buffer) {
  if (threadIdx.x == 0) {
    printf("  stride %d reads buf[%d]:", stride, src);
    for (int j = 0; j < SECTION_SIZE; ++j) {
      printf("%4d", buffer[j]);
    }
    printf("\n");
  }
}

__global__ void kogge_stone_double_buffer_scan(const int* X, int* Y, int n) {
  __shared__ int buf[2][SECTION_SIZE];
  const int t = threadIdx.x;
  const int i = blockIdx.x * SECTION_SIZE + t;
  int src = 0;
  buf[src][t] = (i < n) ? X[i] : 0;

  for (int stride = 1; stride < SECTION_SIZE; stride *= 2) {
    __syncthreads();
    print_buffer_read_by(stride, src, buf[src]);
    int v = buf[src][t];
    if (t >= stride) {
      v += buf[src][t - stride];
    }
    const int dst = 1 - src;
    buf[dst][t] = v;
    src = dst;
  }

  if (i < n) {
    Y[i] = buf[src][t];
  }
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-22s", label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

int main() {
  const std::vector<int> x = {3, 1, 7, 0, 4, 1, 6, 3};
  const int n = static_cast<int>(x.size());

  int* d_x = deviceCopyOf(x);
  int* d_y = deviceCopyOf(std::vector<int>(n));

  printRow("x", x);
  launch(kogge_stone_double_buffer_scan, dim3(1), dim3(SECTION_SIZE), d_x, d_y, n);
  CHECK(cudaGetLastError());
  const std::vector<int> y = hostCopyOf(d_y, n);
  printRow("inclusive", y);

  std::vector<int> expected(n);
  std::inclusive_scan(x.begin(), x.end(), expected.begin());
  const bool pass = (y == expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_x));
  CHECK(cudaFree(d_y));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_kogge_stone_double_buffer.cu -o 03_kogge_stone_double_buffer && ./03_kogge_stone_double_buffer

**What to look at in the output:** Each line shows the buffer that a stride reads, so the rows are one step behind step 2: the stride 2 row `3 4 8 7 4 5 7 9` is what step 2 printed after stride 1. The reads alternate between buf[0] and buf[1]; because each step writes into the other buffer, one barrier per step is enough and the result is the same.

## Step 4 of 7: `04_brent_kung.cu`

Brent-Kung with two elements per thread: the up-sweep builds the reduction tree, the down-sweep fills in the positions it skipped. Every step of both sweeps is printed.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_brent_kung.cu
#include <cstdio>
#include <numeric>
#include <vector>

#include "cuda_compat.h"

constexpr int BK_THREADS = 4;
constexpr int BK_SECTION = 2 * BK_THREADS;

__device__ void print_after_step(const char* sweep, int stride, const int* XY) {
  __syncthreads();
  if (threadIdx.x == 0) {
    printf("  %s %d ", sweep, stride);
    for (int j = 0; j < BK_SECTION; ++j) {
      printf("%4d", XY[j]);
    }
    printf("\n");
  }
}

__global__ void brent_kung_scan(const int* X, int* Y, int n) {
  __shared__ int XY[BK_SECTION];
  const int t = threadIdx.x;
  const int base = blockIdx.x * BK_SECTION;
  XY[t] = (base + t < n) ? X[base + t] : 0;
  XY[t + BK_THREADS] = (base + t + BK_THREADS < n) ? X[base + t + BK_THREADS] : 0;

  for (int stride = 1; stride <= BK_THREADS; stride *= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index < BK_SECTION) {
      XY[index] += XY[index - stride];
    }
    print_after_step("up   stride", stride, XY);
  }

  for (int stride = BK_SECTION / 4; stride > 0; stride /= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index + stride < BK_SECTION) {
      XY[index + stride] += XY[index];
    }
    print_after_step("down stride", stride, XY);
  }

  __syncthreads();
  if (base + t < n) {
    Y[base + t] = XY[t];
  }
  if (base + t + BK_THREADS < n) {
    Y[base + t + BK_THREADS] = XY[t + BK_THREADS];
  }
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-14s", label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

int main() {
  const std::vector<int> x = {3, 1, 7, 0, 4, 1, 6, 3};
  const int n = static_cast<int>(x.size());

  int* d_x = deviceCopyOf(x);
  int* d_y = deviceCopyOf(std::vector<int>(n));

  printRow("x", x);
  launch(brent_kung_scan, dim3(1), dim3(BK_THREADS), d_x, d_y, n);
  CHECK(cudaGetLastError());
  const std::vector<int> y = hostCopyOf(d_y, n);
  printRow("inclusive", y);

  std::vector<int> expected(n);
  std::inclusive_scan(x.begin(), x.end(), expected.begin());
  const bool pass = (y == expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_x));
  CHECK(cudaFree(d_y));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_brent_kung.cu -o 04_brent_kung && ./04_brent_kung

**What to look at in the output:** The up-sweep touches only some positions: after stride 1 positions 1, 3, 5 and 7 hold pair sums (4, 7, 5, 9), and after stride 4 position 7 holds the total 25 while positions 2, 4 and 6 still hold their inputs. The down-sweep fills the gaps: stride 2 puts 5+11 = 16 into position 5, and stride 1 completes positions 2, 4 and 6. Try: remove the down-sweep loop; the result is `3 4 7 11 4 5 6 25`, right only at positions 0, 1, 3 and 7.

## Step 5 of 7: `05_coarsened_scan.cu`

Thread coarsening, 4 threads with 4 elements each: a coalesced load, phase 1 scans each thread's run, phase 2 runs Kogge-Stone over the run totals, phase 3 adds the totals to the left.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_coarsened_scan.cu
#include <cstdio>
#include <numeric>
#include <vector>

#include "cuda_compat.h"

constexpr int CO_THREADS = 4;
constexpr int CO_COARSEN = 4;
constexpr int CO_SECTION = CO_THREADS * CO_COARSEN;

__device__ void print_shared(const char* label, const int* values, int count) {
  __syncthreads();
  if (threadIdx.x == 0) {
    printf("  %-10s", label);
    for (int j = 0; j < count; ++j) {
      printf("%4d", values[j]);
    }
    printf("\n");
  }
}

__device__ void load_section_coalesced(const int* X, int n, int base, int* XY) {
  for (int k = 0; k < CO_COARSEN; ++k) {
    const int j = k * CO_THREADS + threadIdx.x;
    XY[j] = (base + j < n) ? X[base + j] : 0;
  }
}

__device__ void store_section_coalesced(const int* XY, int n, int base, int* Y) {
  for (int k = 0; k < CO_COARSEN; ++k) {
    const int j = k * CO_THREADS + threadIdx.x;
    if (base + j < n) {
      Y[base + j] = XY[j];
    }
  }
}

__device__ void phase1_scan_own_subsection(int* XY, int first) {
  for (int k = 1; k < CO_COARSEN; ++k) {
    XY[first + k] += XY[first + k - 1];
  }
}

__device__ void phase2_kogge_stone_over_subsection_totals(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < CO_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ void phase3_add_total_of_subsections_to_the_left(int* XY, const int* part, int first) {
  const int t = threadIdx.x;
  if (t > 0) {
    const int add = part[t - 1];
    for (int k = 0; k < CO_COARSEN; ++k) {
      XY[first + k] += add;
    }
  }
}

__global__ void coarsened_scan(const int* X, int* Y, int n) {
  __shared__ int XY[CO_SECTION];
  __shared__ int part[CO_THREADS];
  const int t = threadIdx.x;
  const int first = t * CO_COARSEN;
  const int base = blockIdx.x * CO_SECTION;

  load_section_coalesced(X, n, base, XY);
  __syncthreads();

  phase1_scan_own_subsection(XY, first);
  part[t] = XY[first + CO_COARSEN - 1];
  print_shared("phase 1", XY, CO_SECTION);

  phase2_kogge_stone_over_subsection_totals(part);
  print_shared("phase 2", part, CO_THREADS);

  phase3_add_total_of_subsections_to_the_left(XY, part, first);
  print_shared("phase 3", XY, CO_SECTION);

  store_section_coalesced(XY, n, base, Y);
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-10s", label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

int* deviceCopyOf(const std::vector<int>& host) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, host.size() * sizeof(int)));
  CHECK(cudaMemcpy(device, host.data(), host.size() * sizeof(int), cudaMemcpyHostToDevice));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

int main() {
  const std::vector<int> x = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 0, 8, 1, 1, 9, 2};
  const int n = static_cast<int>(x.size());

  int* d_x = deviceCopyOf(x);
  int* d_y = deviceCopyOf(std::vector<int>(n));

  printRow("x", x);
  launch(coarsened_scan, dim3(1), dim3(CO_THREADS), d_x, d_y, n);
  CHECK(cudaGetLastError());
  const std::vector<int> y = hostCopyOf(d_y, n);
  printRow("inclusive", y);

  std::vector<int> expected(n);
  std::inclusive_scan(x.begin(), x.end(), expected.begin());
  const bool pass = (y == expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_x));
  CHECK(cudaFree(d_y));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_coarsened_scan.cu -o 05_coarsened_scan && ./05_coarsened_scan

**What to look at in the output:** Phase 1 restarts the sum every 4 elements: the second run reads 4 5 11 14, the scan of 4 1 6 3 on its own. Phase 2 scans the 4 run totals 11 14 15 13 into 11 25 40 53, and phase 3 adds the total to the left to each run, so the second run becomes 4+11 = 15 up to 14+11 = 25.

## Step 6 of 7: `06_hierarchical_scan.cu`

Any length: scan 4-element blocks and write each block's sum, scan the block sums with the same routine (recursively), then add them back. Every level is printed.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 06_hierarchical_scan.cu
#include <cstdio>
#include <numeric>
#include <vector>

#include "cuda_compat.h"

constexpr int SECTION_SIZE = 4;

__global__ void kogge_stone_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[SECTION_SIZE];
  const int t = threadIdx.x;
  const int i = blockIdx.x * SECTION_SIZE + t;
  XY[t] = (i < n) ? X[i] : 0;

  for (int stride = 1; stride < SECTION_SIZE; stride *= 2) {
    __syncthreads();
    int temp = XY[t];
    if (t >= stride) {
      temp += XY[t - stride];
    }
    __syncthreads();
    XY[t] = temp;
  }

  if (i < n) {
    Y[i] = XY[t];
  }
  if (t == SECTION_SIZE - 1) {
    sums[blockIdx.x] = XY[t];
  }
}

__global__ void add_block_offsets(int* Y, const int* scannedSums, int n) {
  const int i = blockIdx.x * SECTION_SIZE + threadIdx.x;
  if (blockIdx.x > 0 && i < n) {
    Y[i] += scannedSums[blockIdx.x - 1];
  }
}

int* deviceInts(int count) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, count * sizeof(int)));
  return device;
}

std::vector<int> hostCopyOf(const int* device, int count) {
  std::vector<int> host(count);
  CHECK(cudaMemcpy(host.data(), device, count * sizeof(int), cudaMemcpyDeviceToHost));
  return host;
}

void printRow(int level, const char* label, const std::vector<int>& values) {
  std::printf("  level %d  %-12s", level, label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

void printDevice(int level, const char* label, const int* device, int count) {
  printRow(level, label, hostCopyOf(device, count));
}

void scanAnyLength(const int* d_x, int* d_y, int n, int level) {
  const int blocks = static_cast<int>(cdiv(n, SECTION_SIZE));
  int* d_sums = deviceInts(blocks);
  launch(kogge_stone_scan, dim3(blocks), dim3(SECTION_SIZE), d_x, d_y, d_sums, n);
  printDevice(level, "local scans", d_y, n);

  if (blocks > 1) {
    printDevice(level, "block sums", d_sums, blocks);
    int* d_scannedSums = deviceInts(blocks);
    scanAnyLength(d_sums, d_scannedSums, blocks, level + 1);

    launch(add_block_offsets, dim3(blocks), dim3(SECTION_SIZE), d_y, d_scannedSums, n);
    printDevice(level, "add offsets", d_y, n);
    CHECK(cudaFree(d_scannedSums));
  }
  CHECK(cudaFree(d_sums));
}

int main() {
  const std::vector<int> x = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 0, 8, 1, 1, 9, 2, 4, 6, 1};
  const int n = static_cast<int>(x.size());

  int* d_x = deviceInts(n);
  int* d_y = deviceInts(n);
  CHECK(cudaMemcpy(d_x, x.data(), n * sizeof(int), cudaMemcpyHostToDevice));

  printRow(0, "x", x);
  scanAnyLength(d_x, d_y, n, 0);
  CHECK(cudaGetLastError());
  const std::vector<int> y = hostCopyOf(d_y, n);

  std::vector<int> expected(n);
  std::inclusive_scan(x.begin(), x.end(), expected.begin());
  const bool pass = (y == expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_x));
  CHECK(cudaFree(d_y));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 06_hierarchical_scan.cu -o 06_hierarchical_scan && ./06_hierarchical_scan

**What to look at in the output:** Level 0 cuts the 19 values into 5 blocks of 4; the last block holds only 4 6 1, so its sum is 11. The 5 block sums do not fit in one block, so level 1 scans them as two blocks (11 25 40 53 and 11) and level 2 scans its two sums 53 11 into 53 64. On the way back each block adds the scanned sum of the blocks to its left: the fifth block adds 53 and ends at 64, the total. Try: keep only the first 16 values of `x`; the 4 block sums fit in one block and level 1 is the last level.

## Step 7 of 7: `07_single_pass_domino.cu`

One kernel: each block takes a ticket, scans its section, waits for the total of the sections before it, publishes its own total, and stores its elements once.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 07_single_pass_domino.cu
#include <cstdio>
#include <numeric>
#include <vector>

#include "cuda_compat.h"

constexpr int SECTION_SIZE = 4;
constexpr int FLAG_READY = 1;

__device__ void kogge_stone_in_shared(int* XY) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < SECTION_SIZE; stride *= 2) {
    __syncthreads();
    int temp = XY[t];
    if (t >= stride) {
      temp += XY[t - stride];
    }
    __syncthreads();
    XY[t] = temp;
  }
  __syncthreads();
}

__device__ int next_ticket(int* ticket) {
  return atomicAdd(ticket, 1);
}

__device__ int atomic_read(int* address) {
  return atomicAdd(address, 0);
}

__device__ void wait_until_ready(int* flag) {
  int state = atomic_read(flag);
  while (state != FLAG_READY) {
    state = atomic_read(flag);
  }
}

__device__ int wait_for_sum_of_previous_sections(int* flags, int* prefix, int bid) {
  if (bid == 0) {
    return 0;
  }
  wait_until_ready(&flags[bid]);
  return atomic_read(&prefix[bid]);
}

__device__ void publish_sum_of_sections_up_to(int bid, int sum, int* flags, int* prefix) {
  atomicExch(&prefix[bid + 1], sum);
  __threadfence();
  atomicExch(&flags[bid + 1], FLAG_READY);
}

__global__ void domino_scan(const int* X, int* Y, int* flags, int* prefix, int* ticket, int n) {
  __shared__ int XY[SECTION_SIZE];
  __shared__ int bid_s;
  __shared__ int before_s;
  const int t = threadIdx.x;

  if (t == 0) {
    bid_s = next_ticket(ticket);
  }
  __syncthreads();
  const int bid = bid_s;
  const int i = bid * SECTION_SIZE + t;

  XY[t] = (i < n) ? X[i] : 0;
  kogge_stone_in_shared(XY);

  if (t == 0) {
    const int before = wait_for_sum_of_previous_sections(flags, prefix, bid);
    const int sum = before + XY[SECTION_SIZE - 1];
    publish_sum_of_sections_up_to(bid, sum, flags, prefix);
    printf("  block %d got ticket %d: sum before = %3d, publishes %3d\n",
           blockIdx.x,
           bid,
           before,
           sum);
    before_s = before;
  }
  __syncthreads();

  if (i < n) {
    Y[i] = XY[t] + before_s;
  }
}

int* deviceZeros(int count) {
  int* device = nullptr;
  CHECK(cudaMalloc(&device, count * sizeof(int)));
  CHECK(cudaMemset(device, 0, count * sizeof(int)));
  return device;
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-10s", label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

int main() {
  const std::vector<int> x = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 0, 8, 1, 1, 9, 2, 4, 6, 1};
  const int n = static_cast<int>(x.size());
  const int sections = static_cast<int>(cdiv(n, SECTION_SIZE));

  int* d_x = deviceZeros(n);
  int* d_y = deviceZeros(n);
  int* d_flags = deviceZeros(sections + 1);
  int* d_prefix = deviceZeros(sections + 1);
  int* d_ticket = deviceZeros(1);
  CHECK(cudaMemcpy(d_x, x.data(), n * sizeof(int), cudaMemcpyHostToDevice));

  printRow("x", x);
  launch(domino_scan, dim3(sections), dim3(SECTION_SIZE), d_x, d_y, d_flags, d_prefix, d_ticket, n);
  CHECK(cudaGetLastError());
  std::vector<int> y(n);
  CHECK(cudaMemcpy(y.data(), d_y, n * sizeof(int), cudaMemcpyDeviceToHost));
  printRow("inclusive", y);

  std::vector<int> expected(n);
  std::inclusive_scan(x.begin(), x.end(), expected.begin());
  const bool pass = (y == expected);
  std::printf("%s\n", pass ? "PASS" : "FAIL");

  CHECK(cudaFree(d_x));
  CHECK(cudaFree(d_y));
  CHECK(cudaFree(d_flags));
  CHECK(cudaFree(d_prefix));
  CHECK(cudaFree(d_ticket));
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 07_single_pass_domino.cu -o 07_single_pass_domino && ./07_single_pass_domino

**What to look at in the output:** Each line is one block: it receives the total of all earlier sections and publishes its own running total, for example block 2 gets 25 = 11+14 and publishes 25+15 = 40. The published totals 11 25 40 53 64 are the numbers that step 6 needed extra levels and kernels for. On a GPU blocks start in any order, so the ticket and `blockIdx.x` on a line can differ; the section a block scans is the one given by its ticket.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/09_scan/README.md`.

**Next:** open `colab_2_full_programs.ipynb` in this same folder.